In [2]:
# imports & data

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_excel("../data/get_around_delay_analysis.xlsx")
df.shape

(21310, 7)

In [3]:
# diagnostic

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   rental_id                                   21310 non-null  int64  
 1   car_id                                      21310 non-null  int64  
 2   checkin_type                                21310 non-null  object 
 3   state                                       21310 non-null  object 
 4   delay_at_checkout_in_minutes                16346 non-null  float64
 5   previous_ended_rental_id                    1841 non-null   float64
 6   time_delta_with_previous_rental_in_minutes  1841 non-null   float64
dtypes: float64(3), int64(2), object(2)
memory usage: 1.1+ MB


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [5]:
# variables categorical

print(df["state"].value_counts(dropna=False))
print(df["checkin_type"].value_counts(dropna=False))

state
ended       18045
canceled     3265
Name: count, dtype: int64
checkin_type
mobile     17003
connect     4307
Name: count, dtype: int64


In [ ]:
# check for missing values

df.isna().mean().sort_values(ascending=False).round(3)

# 91,4 % des locations n'ont pas de location précédente connue sur la même voiture.

previous_ended_rental_id                      0.914
time_delta_with_previous_rental_in_minutes    0.914
delay_at_checkout_in_minutes                  0.233
rental_id                                     0.000
car_id                                        0.000
checkin_type                                  0.000
state                                         0.000
dtype: float64

In [ ]:
df["delay_at_checkout_in_minutes"].describe()

# Mediane à 9 minutes et moyenne à 59.7 min, avec mas énorme (plusieurs jours..)
# La distribution (écart type..) est écrasée par quelques valeurs extrêmes. Il faudra les traiter.

count    16346.000000
mean        59.701517
std       1002.561635
min     -22433.000000
25%        -36.000000
50%          9.000000
75%         67.000000
max      71084.000000
Name: delay_at_checkout_in_minutes, dtype: float64

In [ ]:
df.groupby("state")["delay_at_checkout_in_minutes"].apply(lambda s: s.isna().mean())

# 9.4% des locations terminées (ended, effective) n'ont pas de retard connu. Il faudra les traiter. 

state
canceled    0.999694
ended       0.094209
Name: delay_at_checkout_in_minutes, dtype: float64

In [ ]:
df[df["state"] == "ended"].groupby("checkin_type")["delay_at_checkout_in_minutes"].apply(lambda s: s.isna().mean())

# la mesure du retard est moins fiable sur mobile (logique, action manuelle).

checkin_type
connect    0.030493
mobile     0.109590
Name: delay_at_checkout_in_minutes, dtype: float64

In [19]:
# nettoyage 

# on garde que les locations effectives (ended) et avec un delay at checkout not null

df_delay = df[
    (df["state"] == "ended")
    & (df["delay_at_checkout_in_minutes"].notna())
].copy()

print("Lignes supprimées :", len(df) - len(df_delay))

Lignes supprimées : 4965


In [20]:
# mesure distributioon des retards pour établir seuil des outliers à ecarter

df_delay["delay_at_checkout_in_minutes"].quantile([0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999])

0.001   -2712.736
0.010    -853.000
0.050    -230.000
0.500       9.000
0.950     397.800
0.990    1490.560
0.999    6858.560
Name: delay_at_checkout_in_minutes, dtype: float64

In [ ]:
# Le PM cherche un délai qui se compte en heures. 
# Au-delà de 12h, la voiture est perdue pour la journée et aucun délai tampon n'y change rien. 
# La distribution confirme la rupture : 25h au 99e centile, 4 jours au 99,9e.

SEUIL = 720 # 12 heures 

In [ ]:
# check impact du seuil 

trop_avance = df_delay["delay_at_checkout_in_minutes"] < -SEUIL
trop_tard   = df_delay["delay_at_checkout_in_minutes"] >  SEUIL

print("Tres en avance :", trop_avance.sum(), f"({trop_avance.mean()*100:.2f} %)")
print("Tres en retard :", trop_tard.sum(),   f"({trop_tard.mean()*100:.2f} %)")

Tres en avance : 243 (1.49 %)
Tres en retard : 494 (3.02 %)


In [ ]:
# 737 lignes écartées, conservées dans df. 
# Le seuil reste une hypothèse, la frontière entre retard réel et défaut de saisie n'est pas observable. 
# Ces cas extrêmes étant les plus pénalisants, on sous-estime légèrement le problème.

In [23]:
# cleaning 

df_delay = df_delay[
    df_delay["delay_at_checkout_in_minutes"].between(-SEUIL, SEUIL)
].copy()

print("Population finale :", len(df_delay))

Population finale : 15608


In [ ]:
# On mesure combien de conducteurs ont réellement attendu leur voiture, et combien de temps.
# 1. Le retard de la location précédente est sur une autre ligne, on va le chercher et on le colle dans une nouvelle colonne (previous_delay_in_minutes).
# 2. Une fois les deux nombres côte à côte, une simple soustraction crée la colonne attente_conducteur.

In [24]:
retards = df_delay[["rental_id", "delay_at_checkout_in_minutes"]].rename(
    columns={
        "rental_id": "previous_ended_rental_id",
        "delay_at_checkout_in_minutes": "previous_delay_in_minutes",
    }
)
retards.head()

,previous_ended_rental_id,previous_delay_in_minutes
1,507750,-81.0
2,508131,70.0
5,511626,-203.0
6,511639,-15.0
7,512303,-44.0


In [25]:
chaines = df[df["previous_ended_rental_id"].notna()].copy()
print("Enchainements :", len(chaines))

Enchainements : 1841


In [26]:
chaines = chaines.merge(retards, on="previous_ended_rental_id", how="left")
chaines[["rental_id", "previous_ended_rental_id", "time_delta_with_previous_rental_in_minutes", "previous_delay_in_minutes"]].head()

,rental_id,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,previous_delay_in_minutes
0,511639,563782.0,570.0,136.0
1,519491,545639.0,420.0,140.0
2,521156,537298.0,0.0,NaN
3,525044,510607.0,60.0,-113.0
4,528808,557404.0,330.0,-352.0


In [27]:
chaines["marge_minutes"] = (
    chaines["previous_delay_in_minutes"]
    - chaines["time_delta_with_previous_rental_in_minutes"]
)

chaines["attente_conducteur"] = chaines["marge_minutes"].clip(lower=0)

print("Retard precedent inconnu  :", chaines["previous_delay_in_minutes"].isna().sum())
print("Conducteurs ayant attendu :", (chaines["attente_conducteur"] > 0).sum())

Retard precedent inconnu  : 168
Conducteurs ayant attendu : 207


In [ ]:
# On a donc 207 conducteur qui attente plus que prévu leurs voiture.
# Verifions le détail des attentes 

chaines[chaines["attente_conducteur"] > 0]["attente_conducteur"].describe()

count    207.000000
mean      60.743961
std       92.965633
min        1.000000
25%       11.000000
50%       25.000000
75%       66.000000
max      596.000000
Name: attente_conducteur, dtype: float64

In [ ]:
# rendre dynamique et tester avec plusieurs type de checkin + seuil

def impact(seuil, scope="all"):
    """Mesure l'effet d'un delai minimum de `seuil` minutes.
    scope : "all" pour toutes les voitures, "connect" pour les Connect uniquement.
    """
    base = chaines if scope == "all" else chaines[chaines["checkin_type"] == "connect"]

    bloquees = base["time_delta_with_previous_rental_in_minutes"] < seuil
    attente = base["attente_conducteur"] > 0

    return {
        "seuil": seuil,
        "scope": scope,
        "locations_bloquees": int(bloquees.sum()),
        "part_bloquee_pct": round(bloquees.mean() * 100, 1),
        "cas_resolus": int((bloquees & attente).sum()),
        "part_resolue_pct": round((bloquees & attente).sum() / attente.sum() * 100, 1),
    }

In [30]:
impact(30)

{'seuil': 30,
 'scope': 'all',
 'locations_bloquees': 279,
 'part_bloquee_pct': np.float64(15.2),
 'cas_resolus': 113,
 'part_resolue_pct': np.float64(54.6)}

In [31]:
import pandas as pd

resultats = pd.DataFrame([
    impact(s, scope)
    for scope in ["all", "connect"]
    for s in [0, 30, 60, 90, 120, 180, 240]
])
resultats

,seuil,scope,locations_bloquees,part_bloquee_pct,cas_resolus,part_resolue_pct
0,0,all,0,0.0,0,0.0
1,30,all,279,15.2,113,54.6
2,60,all,401,21.8,142,68.6
3,90,all,584,31.7,168,81.2
4,120,all,666,36.2,176,85.0
5,180,all,870,47.3,190,91.8
6,240,all,1001,54.4,196,94.7
7,0,connect,0,0.0,0,0.0
8,30,connect,131,16.1,40,58.8
9,60,connect,181,22.3,48,70.6


In [32]:
import plotly.express as px

fig = px.line(
    resultats,
    x="seuil",
    y=["part_bloquee_pct", "part_resolue_pct"],
    facet_col="scope",
    markers=True,
    labels={"seuil": "Delai minimum (min)", "value": "%", "variable": ""},
    title="Compromis entre cas resolus et locations bloquees",
)
fig.show()

/opt/anaconda3/lib/python3.13/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [33]:
df_delay.groupby("checkin_type")["delay_at_checkout_in_minutes"].median()

checkin_type
connect    -8.0
mobile     12.0
Name: delay_at_checkout_in_minutes, dtype: float64

In [ ]:
# Conclusion 

# Un délai minimum de 60 min résoudrait 68,6 % des cas d'attentes en bloquant 21,8 % des locations.
# Au-delà le rendement décroît nettement. 
# Le scope Connect affiche le meilleur rapport bloquée/résolue, ses retards étant structurellement plus courts (médiane -8 min contre +12 sur mobile). 
# Mais c'est précisément là que le problème est le plus faible : le délai minimum traite mal les gros retards, concentrés sur mobile.

In [34]:
# impact CA

pricing = pd.read_csv("../data/get_around_pricing_project.csv")
prix_moyen = pricing["rental_price_per_day"].mean()
print("Prix moyen par jour :", round(prix_moyen, 2), "euros")

Prix moyen par jour : 121.21 euros


In [35]:
PRIX_MOYEN_JOUR = 121.21

def impact_ca(seuil, scope="all"):
    res = impact(seuil, scope)
    res["ca_potentiel_perdu_eur"] = round(res["locations_bloquees"] * PRIX_MOYEN_JOUR)
    return res

impact_ca(60)

{'seuil': 60,
 'scope': 'all',
 'locations_bloquees': 401,
 'part_bloquee_pct': np.float64(21.8),
 'cas_resolus': 142,
 'part_resolue_pct': np.float64(68.6),
 'ca_potentiel_perdu_eur': 48605}

In [36]:
resultats_ca = pd.DataFrame([
    impact_ca(s, scope)
    for scope in ["all", "connect"]
    for s in [0, 30, 60, 90, 120, 180, 240]
])
resultats_ca

,seuil,scope,locations_bloquees,part_bloquee_pct,cas_resolus,part_resolue_pct,ca_potentiel_perdu_eur
0,0,all,0,0.0,0,0.0,0
1,30,all,279,15.2,113,54.6,33818
2,60,all,401,21.8,142,68.6,48605
3,90,all,584,31.7,168,81.2,70787
4,120,all,666,36.2,176,85.0,80726
5,180,all,870,47.3,190,91.8,105453
6,240,all,1001,54.4,196,94.7,121331
7,0,connect,0,0.0,0,0.0,0
8,30,connect,131,16.1,40,58.8,15879
9,60,connect,181,22.3,48,70.6,21939


In [37]:
fig = px.bar(
    resultats_ca,
    x="seuil",
    y="ca_potentiel_perdu_eur",
    color="scope",
    barmode="group",
    labels={"seuil": "Delai minimum (min)", "ca_potentiel_perdu_eur": "CA potentiel perdu (euros)"},
    title="Cout estime du delai minimum, borne haute",
)
fig.show()

In [ ]:
resultats_ca["cout_par_cas_resolu"] = (
    resultats_ca["ca_potentiel_perdu_eur"] / resultats_ca["cas_resolus"]
).round(0)

fig = px.line(
    resultats_ca[resultats_ca["seuil"] > 0],
    x="seuil",
    y="cout_par_cas_resolu",
    color="scope",
    markers=True,
    labels={"seuil": "Delai minimum (min)", "cout_par_cas_resolu": "Euros par cas resolu"},
    title="Cout marginal : combien coute chaque conducteur satisfait",
)
fig.show()

# Connect a le meilleur taux de résolution mais le pire coût par cas résolu. 
# Restreindre à Connect optimise un indicateur de performance, pas l'efficience économique.

In [ ]:
# Limite de l'estimation CA

# Une location bloquée est comptée comme perdue, alors qu'elle peut être simplement reportée.
# Le prix moyen provient du fichier pricing, qui décrit d'autres voitures et ne peut pas être joint au fichier delay.
# Chaque location est comptée comme une journée, sa durée réelle étant inconnue.

In [40]:
# export 

chaines.to_csv("../dashboard/chaines_clean.csv", index=False)
df_delay.to_csv("../dashboard/delay_clean.csv", index=False)

print("chaines   :", len(chaines), "lignes")
print("df_delay  :", len(df_delay), "lignes")

chaines   : 1841 lignes
df_delay  : 15608 lignes
